## 1 - setup and data

In [ ]:
import math, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

SEED_FIT, SEED_TIMING = 7, 3     # the two random streams behind the paper's tables
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, RED = "#2a6f97", "#d9822b", "#b03a2e"

def wilson(k, n, z=1.96):
    """95% Wilson score interval for a proportion k/n."""
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return c - h, c + h

def ztest(k1, n1, k2, n2):
    """Two-proportion z statistic for k1/n1 minus k2/n2 (pooled variance)."""
    p = (k1 + k2) / (n1 + n2)
    return (k1/n1 - k2/n2) / math.sqrt(p*(1-p)*(1/n1 + 1/n2))

pct = lambda x: f"{100*x:.1f}%"

# --- By inning and challenges remaining: (inning, chal, overturned) with 2 left, then with 1 left
INNING = [(1,995,605,37,24),(2,878,548,59,39),(3,903,531,144,87),(4,866,477,184,116),
          (5,849,485,276,145),(6,885,472,349,193),(7,829,469,447,246),(8,724,319,556,283),
          (9,685,250,681,303),(10,34,12,116,52),(11,10,3,31,18),(12,2,1,16,7),(13,0,0,1,0)]
inn = pd.DataFrame(INNING, columns=["inn","n2","k2","n1","k1"]).set_index("inn")
reg = inn.loc[1:9]                # regulation innings; the model covers these only

# --- By role
ROLE = pd.DataFrame({"chal":[5609,4766,182], "ot":[3285,2328,72]},
                    index=["catcher","batter","pitcher"])

# --- By count: balls, strikes, all (chal, ot%), as batter (chal, ot%), defense (chal, ot%)
COUNT = pd.DataFrame([
 (0,0,2417,59.8, 933,55.4,1484,62.6),(0,1,1065,60.8, 449,49.2, 616,69.2),(0,2, 570,51.8, 246,41.1, 324,59.9),
 (1,0, 953,60.4, 436,63.5, 517,57.8),(1,1,1041,54.6, 518,48.1, 523,61.0),(1,2, 884,48.0, 410,40.2, 474,54.6),
 (2,0, 419,58.0, 192,63.5, 227,53.3),(2,1, 678,51.6, 346,46.8, 332,56.6),(2,2, 968,46.7, 468,40.6, 500,52.4),
 (3,0, 137,54.7,  57,57.9,  80,52.5),(3,1, 438,46.6, 216,46.8, 222,46.4),(3,2, 987,41.0, 495,38.4, 492,43.7)],
 columns=["balls","strikes","n_all","pct_all","n_bat","pct_bat","n_def","pct_def"])

# --- By leverage index (page buckets)
LEV = pd.DataFrame([("0.0-0.5 very low",3193,52.4),("0.5-1.0 low",3066,57.7),("1.0-1.5 medium",1806,54.0),
                    ("1.5-2.5 high",1615,53.4),("2.5+ very high",877,46.2)],
                   columns=["bucket","chal","pct"])
LEV["ot"] = (LEV.chal * LEV.pct / 100).round()     # rounded from the page's percentages

# --- By baserunners
BASES = pd.DataFrame([("Bases empty",5531,55.9),("Runner on 1st",2061,53.4),("Runner on 2nd",850,52.4),
                      ("Runner on 3rd",237,49.4),("Runners on 1st & 2nd",896,50.7),("Runners on 1st & 3rd",319,50.2),
                      ("Runners on 2nd & 3rd",260,53.1),("Bases loaded",403,43.9)], columns=["state","chal","pct"])

# --- Challenges per team-game
DIST = pd.Series({0:402,1:1147,2:1592,3:999,4:472,5:167,6:55,7:18,8:3,9:3})

# --- checks
TOTAL = int(ROLE.chal.sum())
assert TOTAL == 10557 and ROLE.ot.sum() == 5685
assert inn.n2.sum() + inn.n1.sum() == TOTAL and inn.k2.sum() + inn.k1.sum() == 5685
assert COUNT.n_all.sum() == TOTAL and COUNT.n_bat.sum() == ROLE.chal["batter"]
assert COUNT.n_def.sum() == ROLE.chal["catcher"] + ROLE.chal["pitcher"]
assert LEV.chal.sum() == TOTAL and BASES.chal.sum() == TOTAL
TEAM_GAMES = int(DIST.sum())
print(f"All tables agree: {TOTAL:,} challenges, {int(ROLE.ot.sum()):,} overturned "
      f"({pct(ROLE.ot.sum()/TOTAL)}), {TEAM_GAMES:,} team-games.")

All tables agree: 10,557 challenges, 5,685 overturned (53.9%), 4,858 team-games.
